In [77]:
import pandas as pd 
import numpy as np
import pickle
import torch
import torch.nn as nn

df = pd.read_csv("data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop = True)

print(df.shape)


(27909, 63)


In [78]:
features = [
     "rolling_avg_fantasy_5", "rolling_avg_fantasy_10", "rolling_std_fantasy_10",
    "rolling_avg_runs_5", "rolling_avg_wickets_5", "batting_position", "matches_played",
    "venue_std_fantasy", "opposition_std_fantasy", "venue_first_appearance",
    "opposition_first_appearance", "won_toss", "expanding_season_fantasy_std",
    "is_home", "role_encoded", "rolling_bowling_contribution_5",
    "rolling_batting_contribution_5", "venue_avg_innings1", "venue_avg_innings2",
    "venue_avg_total_runs", "weather_temp", "weather_humidity", "weather_dew",
    "weather_windspeed", "weather_precip","career_strike_rate"
]

target = "total_fantasy_points"



In [79]:
# cell 2
split_idx = df[df["date"].dt.year >= 2025].index[0]
train = df[:split_idx]
test = df[split_idx:]
X_test = test[features]
y_test = test[target]
print("X_test, y_test built:", X_test.shape, y_test.shape)

X_test, y_test built: (3542, 26) (3542,)


In [80]:
from lightgbm import LGBMRegressor
import numpy as np
print("imported fine")

imported fine


In [81]:
# cell 3
import pickle
with open("models/lgbm_weather.pkl", "rb") as f:
    lgbm = pickle.load(f)
print("model loaded")

model loaded


In [82]:
# cell 4
lgbm_preds = lgbm.predict(X_test)
print("predictions made")

predictions made


In [83]:
lgbm_mae = np.mean(np.abs(lgbm_preds - y_test.values))
print(f"LightGBM MAE (sanity check): {lgbm_mae:.2f}")

LightGBM MAE (sanity check): 21.78


In [84]:
print(df["date"].is_monotonic_increasing)

True


In [85]:
sequence_features = ["runs", "balls_faced", "fours", "sixes", "strike_rate",
    "wickets", "runs_conceded", "balls_bowled", "economy", 
    "maidens", "total_fantasy_points"]

context_features = [
    "venue_avg_fantasy", "venue_std_fantasy",
    "opposition_avg_fantasy", "opposition_std_fantasy",
    "venue_first_appearance", "opposition_first_appearance",
    "is_home", "won_toss", "role_encoded", "batting_position",
    "matches_played", "expanding_season_fantasy_avg",
    "Total_career_runs", "Total_career_wickets",
    "rolling_batting_contribution_5", "rolling_bowling_contribution_5",
    "weather_temp", "weather_humidity", "weather_dew", "weather_windspeed", "weather_precip"
]

SEQ_LEN = 7

X_seq = []
X_context = []
y = []
row_dates = []      # ← track the actual date of each sample
row_indices = []  

for player , group in df.groupby(["player"]):
    player_data = group[sequence_features].values
    context_data = group[context_features].values
    targets = group["total_fantasy_points"].values
    dates = group["date"].values
    indices = group.index.values

    for i in range(len(player_data)):

        if i < SEQ_LEN:
            pad_len = SEQ_LEN - i
            real_data = player_data[:i]
            padding = np.zeros((pad_len, len(sequence_features)))
            seq = np.vstack([padding,real_data])
        
        else:
            seq = player_data[i-SEQ_LEN:i]
        
        X_seq.append(seq)
        X_context.append(context_data[i])
        y.append(targets[i])
        row_dates.append(dates[i])
        row_indices.append(indices[i])


X_seq = np.array(X_seq)
X_context = np.array(X_context)
y = np.array(y)
row_dates = pd.to_datetime(row_dates)
row_indices = np.array(row_indices)

print(X_seq.shape, X_context.shape, y.shape, len(row_dates))

(27909, 7, 11) (27909, 21) (27909,) 27909


In [86]:
test_mask = row_dates.year >= 2025

X_seq_test = X_seq[test_mask]
X_context_test = X_context[test_mask]
y_test_lstm = y[test_mask]
test_indices_lstm = row_indices[test_mask]

print(X_seq_test.shape, X_context_test.shape, y_test_lstm.shape)

(3542, 7, 11) (3542, 21) (3542,)


In [87]:
class CricketLSTM(nn.Module):
    def __init__(self, seq_features, context_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_features,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        self.fc1 = nn.Linear(hidden_size + context_features, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, seq, context):
        lstm_out, (hidden, cell) = self.lstm(seq)
        last_output = lstm_out[:, -1, :]
        combined = torch.cat([last_output, context], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(1)

device = torch.device("mps")

model = CricketLSTM(seq_features=11, context_features=21, hidden_size=32)
model.load_state_dict(torch.load("models/best_lstm.pt", map_location=device))
model = model.to(device)
model.eval()

X_seq_test_t = torch.FloatTensor(X_seq_test).to(device)
X_context_test_t = torch.FloatTensor(X_context_test).to(device)

with torch.no_grad():
    lstm_preds = model(X_seq_test_t, X_context_test_t)
    lstm_preds = lstm_preds.cpu().numpy()

lstm_mae = np.mean(np.abs(lstm_preds - y_test_lstm))
print(f"LSTM MAE (sanity check): {lstm_mae:.2f}")

LSTM MAE (sanity check): 21.26


In [ ]:
# reorder the LSTM test arrays to match the LightGBM test set's exact row order
test_indices_lgbm = test.index.values

# build a lookup from df index -> position in the LSTM test arrays
lstm_index_to_pos = {idx: pos for pos, idx in enumerate(test_indices_lstm)}

# reorder LSTM predictions to match LightGBM's row order
reorder = [lstm_index_to_pos[idx] for idx in test_indices_lgbm]

X_seq_test_aligned = X_seq_test[reorder]
X_context_test_aligned = X_context_test[reorder]
y_test_lstm_aligned = y_test_lstm[reorder]

print(np.array_equal(y_test.values, y_test_lstm_aligned))

True


In [ ]:
X_seq_test_t = torch.FloatTensor(X_seq_test_aligned).to(device)
X_context_test_t = torch.FloatTensor(X_context_test_aligned).to(device)

with torch.no_grad():
    lstm_preds_aligned = model(X_seq_test_t, X_context_test_t)
    lstm_preds_aligned = lstm_preds_aligned.cpu().numpy()

lstm_mae_check = np.mean(np.abs(lstm_preds_aligned - y_test_lstm_aligned))
print(f"LSTM MAE (aligned, sanity check): {lstm_mae_check:.2f}")

LSTM MAE (aligned, sanity check): 21.26


In [ ]:
# spot check a few individual rows
for i in range(5):
    idx = test_indices_lgbm[i]
    print("LightGBM row df index:", idx, "-> actual points:", y_test.values[i])
    
    pos_in_lstm_arrays = lstm_index_to_pos[idx]
    print("Same df index in LSTM arrays -> actual points:", y_test_lstm[pos_in_lstm_arrays])
    print("After reorder, same position in aligned array:", y_test_lstm_aligned[i])
    print()

LightGBM row df index: 24367 -> actual points: 4
Same df index in LSTM arrays -> actual points: 4
After reorder, same position in aligned array: 4

LightGBM row df index: 24368 -> actual points: 40
Same df index in LSTM arrays -> actual points: 40
After reorder, same position in aligned array: 40

LightGBM row df index: 24369 -> actual points: 40
Same df index in LSTM arrays -> actual points: 40
After reorder, same position in aligned array: 40

LightGBM row df index: 24370 -> actual points: 32
Same df index in LSTM arrays -> actual points: 32
After reorder, same position in aligned array: 32

LightGBM row df index: 24371 -> actual points: 10
Same df index in LSTM arrays -> actual points: 10
After reorder, same position in aligned array: 10



In [ ]:
ensemble_preds = 0.5 * lgbm_preds + 0.5 * lstm_preds_aligned
ensemble_mae = np.mean(np.abs(ensemble_preds - y_test.values))
print(f"Ensemble MAE (50/50): {ensemble_mae:.2f}")

Ensemble MAE (50/50): 20.99


In [ ]:
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    blend = w * lgbm_preds + (1 - w) * lstm_preds_aligned
    mae = np.mean(np.abs(blend - y_test.values))
    print(f"LightGBM weight {w}: MAE = {mae:.2f}")

LightGBM weight 0.3: MAE = 21.05
LightGBM weight 0.4: MAE = 21.01
LightGBM weight 0.5: MAE = 20.99
LightGBM weight 0.6: MAE = 21.00
LightGBM weight 0.7: MAE = 21.02


In [ ]:
import pickle
import json 

with open("models/lgbm_final.pkl", "wb") as f:
    pickle.dump(lgbm, f)

torch.save(model.state_dict(), "models/lstm_final.pt")

ensemble_config = {
    "lgbm_weight": 0.5,
    "lstm_weight": 0.5,
    "features_lgbm": features,
    "sequence_features": sequence_features,
    "context_features": context_features,
    "seq_len": SEQ_LEN,
    "verified_test_mae": 20.99
}

with open("models/ensemble_config.json", "w") as f:
    json.dump(ensemble_config, f, indent=2)

print("Saved final ensemble artifacts")

Saved final ensemble artifacts
